# **Segregation of the Dataset and Mean Difference Analysis**

## _ETL Workflow Notebook 5_

## Content:
1. Calculating general statistics for a given column; 
2. Getting data quantiles for a given column; 
3. Getting a particular P-percent quantile limit for a given column;  
4. Selecting subsets from a dataframe (using row filters) and labelling these subsets;
5. Estimating ideal sample size for tests;
6. Performing Analysis of Variance (ANOVA); and obtaining box plots or violin plots;
7. Performing AB-Tests.

Marco Cesar Prado Soares, Data Scientist Specialist - Bayer Crop Science LATAM
- marcosoares.feq@gmail.com
- marco.soares@bayer.com

## **Load Python Libraries in Global Context**

In [ ]:
import load
from idsw import *

## **Call the functions**

### **Importing the dataset**

In [ ]:
## WARNING: Use this function to load dataframes stored on Excel (xls, xlsx, xlsm, xlsb, odf, ods and odt), 
## JSON, txt, or CSV (comma separated values) files. Tables in webpages or html files can also be read.

FILE_DIRECTORY_PATH = ""
# FILE_DIRECTORY_PATH - (string, in quotes): input the path of the directory 
# (e.g. folder path) where the file is stored. e.g. FILE_DIRECTORY_PATH = "" 
# or FILE_DIRECTORY_PATH = "folder"

FILE_NAME_WITH_EXTENSION = "dataset.csv"
# FILE_NAME_WITH_EXTENSION - (string, in quotes): input the name of the file with the 
# extension. e.g. FILE_NAME_WITH_EXTENSION = "file.xlsx", or, 
# FILE_NAME_WITH_EXTENSION = "file.csv", "file.txt", or "file.json"
# Again, the extensions may be: xls, xlsx, xlsm, xlsb, odf, ods, odt, json, txt or csv.
# Also, html files and webpages may be also read.

# You may input the path for an HTML file containing a table to be read; or 
# a string containing the address for a webpage containing the table. The address must start
# with www or htpp. If a website is input, the full address can be input as FILE_DIRECTORY_PATH
# or as FILE_NAME_WITH_EXTENSION.

LOAD_TXT_FILE_WITH_JSON_FORMAT = False
# LOAD_TXT_FILE_WITH_JSON_FORMAT = False. Set LOAD_TXT_FILE_WITH_JSON_FORMAT = True 
# if you want to read a file with txt extension containing a text formatted as JSON 
# (but not saved as JSON).
# WARNING: if LOAD_TXT_FILE_WITH_JSON_FORMAT = True, all the JSON file parameters of the 
# function (below) must be set. If not, an error message will be raised.

HOW_MISSING_VALUES_ARE_REGISTERED = None
# HOW_MISSING_VALUES_ARE_REGISTERED = None: keep it None if missing values are registered as None,
# empty or np.nan. Pandas automatically converts None to NumPy np.nan objects (floats).
# This parameter manipulates the argument na_values (default: None) from Pandas functions.
# By default the following values are interpreted as NaN: ‘’, ‘#N/A’, ‘#N/A N/A’, ‘#NA’, ‘-1.#IND’, 
#‘-1.#QNAN’, ‘-NaN’, ‘-nan’, ‘1.#IND’, ‘1.#QNAN’, ‘<NA>’, ‘N/A’, ‘NA’, ‘NULL’, ‘NaN’, 
# ‘n/a’, ‘nan’, ‘null’.

# If a different denomination is used, indicate it as a string. e.g.
# HOW_MISSING_VALUES_ARE_REGISTERED = '.' will convert all strings '.' to missing values;
# HOW_MISSING_VALUES_ARE_REGISTERED = 0 will convert zeros to missing values.

# If dict passed, specific per-column NA values. For example, if zero is the missing value
# only in column 'numeric_col', you can specify the following dictionary:
# how_missing_values_are_registered = {'numeric-col': 0}

    
HAS_HEADER = True
# HAS_HEADER = True if the the imported table has headers (row with columns names).
# Alternatively, HAS_HEADER = False if the dataframe does not have header.

DECIMAL_SEPARATOR = '.'
# DECIMAL_SEPARATOR = '.' - String. Keep it '.' or None to use the period ('.') as
# the decimal separator. Alternatively, specify here the separator.
# e.g. DECIMAL_SEPARATOR = ',' will set the comma as the separator.
# It manipulates the argument 'decimal' from Pandas functions.

TXT_CSV_COL_SEP = "comma"
# txt_csv_col_sep = "comma" - This parameter has effect only when the file is a 'txt'
# or 'csv'. It informs how the different columns are separated.
# Alternatively, txt_csv_col_sep = "comma", or txt_csv_col_sep = "," 
# for columns separated by comma;
# txt_csv_col_sep = "whitespace", or txt_csv_col_sep = " " 
# for columns separated by simple spaces.
# You can also set a specific separator as string. For example:
# txt_csv_col_sep = '\s+'; or txt_csv_col_sep = '\t' (in this last example, the tabulation
# is used as separator for the columns - '\t' represents the tab character).

## Parameters for loading Excel files:

LOAD_ALL_SHEETS_AT_ONCE = False
# LOAD_ALL_SHEETS_AT_ONCE = False - This parameter has effect only when for Excel files.
# If LOAD_ALL_SHEETS_AT_ONCE = True, the function will return a list of dictionaries, each
# dictionary containing 2 key-value pairs: the first key will be 'sheet', and its
# value will be the name (or number) of the table (sheet). The second key will be 'df',
# and its value will be the pandas dataframe object obtained from that sheet.
# This argument has preference over SHEET_TO_LOAD. If it is True, all sheets will be loaded.
    
SHEET_TO_LOAD = None
# SHEET_TO_LOAD - This parameter has effect only when for Excel files.
# keep SHEET_TO_LOAD = None not to specify a sheet of the file, so that the first sheet
# will be loaded.
# SHEET_TO_LOAD may be an integer or an string (inside quotes). SHEET_TO_LOAD = 0
# loads the first sheet (sheet with index 0); SHEET_TO_LOAD = 1 loads the second sheet
# of the file (index 1); SHEET_TO_LOAD = "Sheet1" loads a sheet named as "Sheet1".
# Declare a number to load the sheet with that index, starting from 0; or declare a
# name to load the sheet with that name.

## Parameters for loading JSON files:

JSON_RECORD_PATH = None
# JSON_RECORD_PATH (string): manipulate parameter 'record_path' from json_normalize method.
# Path in each object to list of records. If not passed, data will be assumed to 
# be an array of records. If a given field from the JSON stores a nested JSON (or a nested
# dictionary) declare it here to decompose the content of the nested data. e.g. if the field
# 'books' stores a nested JSON, declare, JSON_RECORD_PATH = 'books'

JSON_FIELD_SEPARATOR = "_"
# JSON_FIELD_SEPARATOR = "_" (string). Manipulates the parameter 'sep' from json_normalize method.
# Nested records will generate names separated by sep. 
# e.g., for JSON_FIELD_SEPARATOR = ".", {‘foo’: {‘bar’: 0}} -> foo.bar.
# Then, if a given field 'main_field' stores a nested JSON with fields 'field1', 'field2', ...
# the name of the columns of the dataframe will be formed by concatenating 'main_field', the
# separator, and the names of the nested fields: 'main_field_field1', 'main_field_field2',...

JSON_METADATA_PREFIX_LIST = None
# JSON_METADATA_PREFIX_LIST: list of strings (in quotes). Manipulates the parameter 
# 'meta' from json_normalize method. Fields to use as metadata for each record in resulting 
# table. Declare here the non-nested fields, i.e., the fields in the principal JSON. They
# will be repeated in the rows of the dataframe to give the metadata (context) of the rows.

# e.g. Suppose a JSON with the following structure: {'name': 'Mary', 'last': 'Shelley',
# 'books': [{'title': 'Frankestein', 'year': 1818}, {'title': 'Mathilda ', 'year': 1819},{'title': 'The Last Man', 'year': 1826}]},
# Here, there are nested JSONs in the field 'books'. The fields that are not nested
# are 'name' and 'last'.
# Then, JSON_RECORD_PATH = 'books'
# JSON_METADATA_PREFIX_LIST = ['name', 'last']


# The dataframe will be stored in the object named 'dataset':
# Simply modify this object on the left of equality:
dataset = load_pandas_dataframe (file_directory_path = FILE_DIRECTORY_PATH, file_name_with_extension = FILE_NAME_WITH_EXTENSION, load_txt_file_with_json_format = LOAD_TXT_FILE_WITH_JSON_FORMAT, how_missing_values_are_registered = HOW_MISSING_VALUES_ARE_REGISTERED, has_header = HAS_HEADER, decimal_separator = DECIMAL_SEPARATOR, txt_csv_col_sep = TXT_CSV_COL_SEP, load_all_sheets_at_once = LOAD_ALL_SHEETS_AT_ONCE, sheet_to_load = SHEET_TO_LOAD, json_record_path = JSON_RECORD_PATH, json_field_separator = JSON_FIELD_SEPARATOR, json_metadata_prefix_list = JSON_METADATA_PREFIX_LIST)

# OBS: If an Excel file is loaded and LOAD_ALL_SHEETS_AT_ONCE = True, then the object
# dataset will be a list of dictionaries, with 'sheet' as key containing the sheet name; and 'df'
# as key correspondent to the Pandas dataframe. So, to access the 3rd dataframe (index 2, since
# indexing starts from zero): df = dataframe[2]['df'], where dataframe is the list returned.

### **Calculating general statistics for a given column**

In [ ]:
DATASET = dataset #Alternatively: object containing the dataset to be analyzed

COLUMN_TO_ANALYZE = 'column_to_analyze'
# COLUMN_TO_ANALYZE = string (inside quotes) containing the name of the column that will be analyzed. 
# e.g. column_to_analyze = "column1" will analyze the column named as 'column1'.

# Statistics dataframe saved as general_stats. 
# Simply modify this object on the left of equality:
general_stats = column_general_statistics (df = DATASET, column_to_analyze = COLUMN_TO_ANALYZE)

### **Getting data quantiles for a given column**

In [ ]:
DATASET = dataset #Alternatively: object containing the dataset to be analyzed

COLUMN_TO_ANALYZE = 'column_to_analyze'
# COLUMN_TO_ANALYZE = string (inside quotes) containing the name of the column that will be analyzed. 
# e.g. column_to_analyze = "column1" will analyze the column named as 'column1'.

# Quantiles dataframe saved as quantiles_summ_df
# Simply modify this object on the left of equality:
quantiles_summ_df = get_quantiles_for_column (df = DATASET, column_to_analyze = COLUMN_TO_ANALYZE)

### **Getting a particular P-percent quantile limit for a given column**
- input P as a percent, from 0 to 100.

In [ ]:
DATASET = dataset #Alternatively: object containing the dataset to be analyzed

COLUMN_TO_ANALYZE = 'column_to_analyze'
# COLUMN_TO_ANALYZE = string (inside quotes) containing the name of the column that will be analyzed. 
# e.g. column_to_analyze = "column1" will analyze the column named as 'column1'.

P_PERCENT = 100
# P_PERCENT: float value from 0 to 100 representing the percent of the quantile
# if P_PERCENT = 31.2, then 31.2% of the data will fall below the returned value
# if P_PERCENT = 75, then 75% of the data will fall below the returned value
# if P_PERCENT = 0, the minimum value is returned.
# if P_PERCENT = 100, the maximum value is returned.

# P-Percent quantile limit returned as the float value quantile_lim
# Simply modify this variable on the left of equality:
quantile_lim = get_p_percent_quantile_lim_for_column (df = DATASET, column_to_analyze = COLUMN_TO_ANALYZE, p_percent = P_PERCENT)

### **Selecting subsets from a dataframe (using row filters) and labelling these subsets**

In [ ]:
# Attention: this function selects subsets from the dataframe and label them, 
# allowing the seggregation of the data.
# If you want to filter the dataframe to eliminate non-selected rows, use the 
# function apply_row_filters_list
    
DATASET = dataset #Alternatively: object containing the dataset to be analyzed

LIST_OF_LABELS = [
    
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}, 
    {'filter': None, 
     'value_to_apply': None, 'new_column_name': None}
    
]

# LIST_OF_LABELS = [{'filter': None, 'value_to_apply': None, 'new_column_name': None}]
# LIST_OF_LABELS is as a list of dictionaries. It must be declared as a list, in brackets,
# even if there is a single dictionary.
# Use always the same keys: 'filter' for one of the boolean filters that will be applied; 
# 'value_to_apply' the value that will be used for labelling that specific subset selected
# from the boolean filter (it may be either a string or a value); and
# 'new_column_name': a string or variable to be the name of the new column created. If None,
# a standard name will be applied.

## ATTENTION: If you want the labels to be applied to a same column, declare the exact same value
# for the key 'new_column_name'. Also, if you want the value to be applied to an existing column,
# declare the existing column's name in 'new_column_name'.
    
# If you want, you can remove elements (dictionaries) from the list to declare fewer elements;
# and you can also add more elements (dictionaries) to the lists, if you need to plot more series.
# Simply put a comma after the last element from the list and declare a new dictionary, keeping the
# same keys: {'filter': filter, 'value_to_apply': value, 'new_column_name': name}, where 
# filter, value, and name represent the boolean filter, the value for labelling, and the new
# column name (you can pass 'value_to_apply': None, 'new_column_name': None, but if 
# 'filter' is None, the new dictionary will be ignored).

## define the filters and only them define the filters list
# EXAMPLES OF BOOLEAN FILTERS TO COMPOSE THE LIST
# boolean_filter1 = ((None) & (None)) # (condition1 and (&) condition2)
# boolean_filter2 = ((None) | (None)) # condition1 or (|) condition2
# boolean filters result into boolean values True or False.

## Examples of filters:
## filter1 = (condition 1) & (condition 2)
## filter1 = (df['column1'] > = 0) & (df['column2']) < 0)
## filter2 = (condition)
## filter2 = (df['column3'] <= 2.5)
## filter3 = (df['column4'] > 10.7)
## filter3 = (condition 1) | (condition 2)
## filter3 = (df['column5'] != 'string1') | (df['column5'] == 'string2')
    
## comparative operators: > (higher); >= (higher or equal); < (lower); 
## <= (lower or equal); == (equal); != (different)
    
## concatenation operators: & (and): the filter is True only if the 
## two conditions concatenated through & are True
## | (or): the filter is True if at least one of the two conditions concatenated
## through | are True.
## ~ (not): inverts the boolean, i.e., True becomes False, and False becomes True. 
    
## separate conditions with parentheses. Use parentheses to define a order
## of definition of the conditions:
## filter = ((condition1) & (condition2)) | (condition3)
## Here, firstly ((condition1) & (condition2)) = subfilter is evaluated. 
## Then, the resultant (subfilter) | (condition3) is evaluated.

## Pandas .isin method: you can also use this method to filter rows belonging to
## a given subset (the row that is in the subset is selected). The syntax is:
## is_black_or_brown = dogs["color"].isin(["Black", "Brown"])
## or: filter = (dataframe_column_series).isin([value1, value2, ...])
# The negative of this condition may be acessed with ~ operator:
##  filter = ~(dataframe_column_series).isin([value1, value2, ...])
## Also, you may use isna() method as filter for missing values:
## filter = (dataframe_column_series).isna()
## or, for not missing: ~(dataframe_column_series).isna()

# Labelled dataframe saved as labelled_df
# Simply modify this object on the left of equality:
labelled_df = label_dataframe_subsets (df = DATASET, list_of_labels = LIST_OF_LABELS)

### **Estimating ideal sample size for tests**
- Evaluate the sample size required to get a significant result out of your **AB or ANOVA test**.
    - **Example**: you are working for a food delivery app to determine if a new feature which provides custom suggestions to each user based on their preferences will increase the `conversion rate` of the app. This rate measures the rate of users who "converted" or placed an order using the app. **The current CVR of the app is 12% and the stakeholders would like the new feature to increase it up to a 14%**. Given this expectation you can compute the required sample size to perform this analysis.
    
    
- For comparison of proportions of success on binary (categorical) results, the compared column must have been labelled with 0 for failure and 1 for success.

In [ ]:
TARGET_MEAN_OR_PROPORTION = 0.14
# TARGET_MEAN_OR_PROPORTION is a numeric (real) value correspondent to the mean or proportion that 
# the user wants to observe. If it is a proportion, p must be input as a fraction. 
# For example, if the user wants to perform a test to verify a modification of conversion rate from 
# 12 to 14%, then the target proportion is 0.14, i.e., TARGET_MEAN_OR_PROPORTION = 0.14.
# If the target is a mean, any real number may be input. For instance, if user wants to verify a 
# decrease of 200 to 150ºC of temperature, TARGET_MEAN_OR_PROPORTION = 150.

CURRENT_MEAN_OR_PROPORTION = None
# CURRENT_MEAN_OR_PROPORTION is a numeric value correspondent to the mean or proportion that the user 
# observes now. User may directly input it or the value can be retrieved from the response column on 
# the dataset. To calculate from the response on dataset, keep CURRENT_MEAN_OR_PROPORTION = None

# IN CASE OF INPUT OF PARAMETER CURRENT_MEAN_OR_PROPORTION
# If it is a proportion, p must be input as a fraction. For example, if the user wants to perform a 
# test to verify a modification of conversion rate from 12 to 14%, then the 
# current proportion is 0.12, i.e., CURRENT_MEAN_OR_PROPORTION = 0.12.
# If the target is a mean, any real number may be input. For instance, if user wants to verify a 
# decrease of 200 to 150ºC of temperature, CURRENT_MEAN_OR_PROPORTION = 200.

CURRENT_STANDARD_DEVIATION = None
# CURRENT_STANDARD_DEVIATION = None - numeric (real) value representing 
# the current standard deviation of the response variable.
# If user wants to retrieve the information from the response column on the dataset, 
# CURRENT_STANDARD_DEVIATION is kept as None.
# If user wants to input the current mean value as current_mean_or_proportion (manual input) 
# to check difference of mean, a real number representing the current standard deviation must be 
# also input. Example: CURRENT_STANDARD_DEVIATION = 2.51
# Notice that this parameter only matters for the tests of mean difference, and is ignored 
# when evaluating different proportions.
# If no value is provided, but a column and a dataset are pointed, then the function will try to 
# retrieve the value from the dataframe, even if a manual value of mean is provided.
    
    
WHAT_TO_TEST = 'proportion'
# WHAT_TO_TEST = 'mean' for verifying differences between mean values.
# WHAT_TO_TEST = 'proportion' for verifying differences between proportions.

CONFIDENCE_LEVEL_PERCENT = 95
# CONFIDENCE_LEVEL_PERCENT = 95 = 95% confidence
# It is the percent of confidence for the analysis.
# Set CONFIDENCE_LEVEL_PERCENT = 90 to get 0.90 = 90% confidence in the analysis.
# Notice that, when less trust is needed, we can reduce confidence_level_pct
# to get less restrictive results.
TEST_POWER_PERCENT = 20
# TEST_POWER_PERCENT = 20 = 20% power
# It is the percent of power of the test.
# Set TEST_POWER_PERCENT = 20 to get 0.20 = 20% power in the analysis.
# Notice that, when less trust is needed, we can reduce the power to get less restrictive results.
PERFORM_TWO_SIDED_TEST = True
# PERFORM_TWO_SIDED_TEST = True. Wether or not to perform a two-sided test. Keep it
# True to use the 2-sided test, which will evaluate if there is a significant difference (higher
# or lower). This is the recommended configuration. 
# If PERFORM_TWO_SIDED_TEST = False, a one-sided test will be used instead (not recommended configuration).

DATASET = None
VARIABLE_TO_ANALYZE = None
# IN CASE A MANUAL INPUT OF CURRENT MEAN OR PROPORTION WAS NOT PROVIDED:
# A dataframe object must be input as DATASET. Example: DATASET = df
# Also the column containing the current values of the variable to analyze must be provided as 
# VARIABLE_TO_ANALYZE.
# DATASET is an object, so do not declare it in quotes. 
# VARIABLE_TO_ANALYZE is a string, so declare it in quotes.
# Example: VARIABLE_TO_ANALYZE = 'response'
# If manual inputs are provided, you may keep DATASET = None and VARIABLE_TO_ANALYZE = None


# Variable storing the sample size was returned as required_sample_size
# Simply modify this variable on the left of equality:
required_sample_size = estimate_sample_size (target_mean_or_proportion = TARGET_MEAN_OR_PROPORTION, current_mean_or_proportion = CURRENT_MEAN_OR_PROPORTION, current_standard_deviation = CURRENT_STANDARD_DEVIATION, what_to_test = WHAT_TO_TEST, confidence_level_pct = CONFIDENCE_LEVEL_PERCENT, test_power_pct = TEST_POWER_PERCENT, perform_two_sided_test = PERFORM_TWO_SIDED_TEST, df = DATASET, variable_to_analyze = VARIABLE_TO_ANALYZE)

### **Performing Analysis of Variance (ANOVA); and for obtaining box plots or violin plots**

In [ ]:
PLOT_TYPE = 'box'
# PLOT_TYPE = 'box' to plot a boxplot.
# PLOT_TYPE = 'violin' to plot a violinplot.
# If PLOT_TYPE = None, or PLOT_TYPE = 'only_anova', only the anova analysis will be performed.

CONFIDENCE_LEVEL_PERCENT = 95
# CONFIDENCE_LEVEL_PERCENT = 95 = 95% confidence
# It is the percent of confidence for the analysis.
# Set CONFIDENCE_LEVEL_PERCENT = 90 to get 0.90 = 90% confidence in the analysis.
# Notice that, when less trust is needed, we can reduce confidence_level_pct
# to get less restrictive results.

ORIENTATION = 'vertical'
# ORIENTATION = 'vertical' for vertical plots; 
# ORIENTATION = 'horizontal', for horizontal plots.
    
REFERENCE_VALUE = None
# REFERENCE_VALUE: keep it as None or add a float value.
# This reference value will be shown as a red constant line to be compared
# with the plots. e.g. REFERENCE_VALUE = 1.0 will plot a red line passing through
# VARIABLE_TO_ANALYZE = 1.0

DATA_IN_SAME_COLUMN = False

# Parameters to input when DATA_IN_SAME_COLUMN = True:
DATASET = None #Alternatively: object containing the dataset to be analyzed (e.g. DATASET = dataset)
COLUMN_WITH_LABELS_OR_GROUPS = None # Alternatively: correct name for X-column
VARIABLE_TO_ANALYZE = None # Alternatively: correct name for Y-column

# DATA_IN_SAME_COLUMN = False: set as True if all the values to plot are in a same column.
# If DATA_IN_SAME_COLUMN = True, you must specify the dataframe containing the data as DATASET;
# the column containing the label or group indication as COLUMN_WITH_LABELS_OR_GROUPS; and the column 
# containing the variable to analyze as VARIABLE_TO_ANALYZE.

# If COLUMN_WITH_LABELS_OR_GROUPS is None, the ANOVA analysis will not be performed and 
# the plot will be obtained for the whole series.

# DATASET is an object, so do not declare it in quotes. The other three arguments (columns' names) 
# are strings, so declare in quotes.     
# Example: suppose you have a dataframe saved as dataset, and two groups A and B to compare. 
# All the results for both groups are in a column named 'results'. If the result is for
# an entry from group A, then a column named 'group' has the value 'A'. If it is for group B,
# column 'group' shows the value 'B'. In this example:
# DATA_IN_SAME_COLUMN = True,
# DATASET = dataset,
# COLUMN_WITH_LABELS_OR_GROUPS = 'group',
# VARIABLE_TO_ANALYZE = 'results'.


# If you want to declare a list of dictionaries, keep DATA_IN_SAME_COLUMN = False and keep
# DATASET = None (the other arguments may be set as None, but it is not mandatory: 
# COLUMN_WITH_LABELS_OR_GROUPS = None, VARIABLE_TO_ANALYZE = None).
    
# Parameter to input when DATA_IN_SAME_COLUMN = False:
LIST_OF_DICTIONARIES_WITH_SERIES_TO_ANALYZE = [
    
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}, 
    
]
# LIST_OF_DICTIONARIES_WITH_SERIES_TO_ANALYZE: {'values_to_analyze': None, 'label': None}
# if data is already converted to series, lists or arrays, provide them as a list of dictionaries. 
# It must be declared as a list, in brackets, even if there is a single dictionary.
# Use always the same keys: 'values_to_analyze' for values that will be analyzed, and 'label' for
# the label or group correspondent to the series (may be a number or a string). 
# If you do not want to declare a series, simply keep as None, but do not remove or rename a 
# key (ALWAYS USE THE KEYS SHOWN AS MODEL).
# If you want, you can remove elements (dictionaries) from the list to declare fewer elements;
# and you can also add more elements (dictionaries) to the lists, if you need to plot more series.
# Simply put a comma after the last element from the list and declare a new dictionary, keeping the
# same keys: {'values_to_analyze': y, 'label': 'series_y'}, where y represents the values
# to analyze, and 'series_y' is the label 
# (you can pass 'label': None, but if values_to_analyze' is None, the new 
# dictionary will be ignored).
    
# Examples:
# LIST_OF_DICTIONARIES_WITH_SERIES_TO_ANALYZE = [{'values_to_analyze': y, 'label': 0}]
# will plot a single variable. In turns: LIST_OF_DICTIONARIES_WITH_SERIES_TO_ANALYZE = 
# [{'values_to_analyze': DATASET['Y1'], 'label': 'label1'}, 
# {'values_to_analyze': DATASET['Y2'], 'label': 'label2'}, {'x': None, 'y': None, 'lab': None}, {'x': None, 'y': None, 'lab': None}, {'x': None, 'y': None, 'lab': None}, {'x': None, 'y': None, 'lab': None}, {'x': None, 'y': None, 'lab': None}, {'x': None, 'y': None, 'lab': None}, {'x': None, 'y': None, 'lab': None}, {'x': None, 'y': None, 'lab': None}, {'x': None, 'y': None, 'lab': None}]
# will plot two series, Y1 and Y2.
# Notice that all dictionaries where 'values_to_analyze' is None are automatically ignored.
# If None is provided to 'label', an automatic label will be generated.

## Parameters with effect only for boxplots (PLOT_TYPE = 'box'):
OBTAIN_BOXPLOT_WITH_FILLED_BOXES = True
# Manipulate parameter patch_artist (boolean, default: False)    
# If OBTAIN_BOXPLOT_WITH_FILLED_BOXES = True, the boxes are created filled.     
# If OBTAIN_BOXPLOT_WITH_FILLED_BOXES = False, only the contour of the boxes are shown    
# (obtain void white boxes).
OBTAIN_BOXPLOT_WITH_NOTCHED_BOXES = False
# OBTAIN_BOXPLOT_WITH_NOTCHED_BOXES = False    
# Manipulate parameter notch (boolean, default: False) from the boxplot object    
# Whether to draw a notched boxplot (OBTAIN_BOXPLOT_WITH_NOTCHED_BOXES = True), 
# or a rectangular boxplot (OBTAIN_BOXPLOT_WITH_NOTCHED_BOXES = False).     
# The notches represent the confidence interval (CI) around the median.

X_AXIS_ROTATION = 0
#Rotation of X axis labels. Alternatively, insert any numeric value from 0 to 90 (degrees).
Y_AXIS_ROTATION = 0
#Rotation of Y axis labels. Alternatively, insert any numeric value from 0 to 90 (degrees).
GRID = True #Alternatively: True or False
# If GRID = False, no grid lines are shown in the graphic.
HORIZONTAL_AXIS_TITLE = None #Alternatively: string inside quotes for horizontal title
VERTICAL_AXIS_TITLE = None #Alternatively: string inside quotes for vertical title
PLOT_TITLE = None #Alternatively: string inside quotes for graphic title
# e.g. HORIZONTAL_AXIS_TITLE = "X", VERTICAL_AXIS_TITLE = "Y", PLOT_TITLE = "YxX"

EXPORT_PNG = False
# Set EXPORT_PNG = False if you do not want to export the obtained image;
# Set EXPORT_PNG = True to export the obtained image.
DIRECTORY_TO_SAVE = None
# This parameter has effect only if EXPORT_PNG = True.
# (string, in quotes): input the path of the directory 
# (e.g. folder path) where the file will be stored. e.g. DIRECTORY_TO_SAVE = "" 
# or DIRECTORY_TO_SAVE = "folder"
# If EXPORT_PNG = True and DIRECTORY_TO_SAVE = None, the file will be saved in the root
# path, DIRECTORY_TO_SAVE = ""
FILE_NAME = None
# This parameter has effect only if EXPORT_PNG = True.
# (string, in quotes): input the name you want for the file without the 
# extension, which will be 'png'. e.g. FILE_NAME = "my_image" will save a file 'my_image.png' 
# If EXPORT_PNG = True and FILE_NAME = None, the file will be saved as:
# '{PLOT_TYPE}_plot.png'
# WARNING: if there is already a file in the path DIRECTORY_TO_SAVE saved as FILE_NAME,
# the file will be overwritten.
PNG_RESOLUTION_DPI = 330
# This parameter has effect only if EXPORT_PNG = True.
# Alternatively, input an integer that will correspond to the resolution of the exported
# image in dpi. If PNG_RESOLUTION_DPI = None, it will be set as 330.


# Dictionary storing ANOVA F-test and p-value returned as anova_summary_dict
# Dictionary mapping each component of the box or violin plot returned as plot_returned_dict
# Simply modify these objects on the left of equality:
anova_summary_dict, plot_returned_dict = anova_box_violin_plot (plot_type = PLOT_TYPE, confidence_level_pct = CONFIDENCE_LEVEL_PERCENT, orientation = ORIENTATION, reference_value = REFERENCE_VALUE, data_in_same_column = DATA_IN_SAME_COLUMN, df = DATASET, column_with_labels_or_groups = COLUMN_WITH_LABELS_OR_GROUPS, variable_to_analyze = VARIABLE_TO_ANALYZE, list_of_dictionaries_with_series_to_analyze = LIST_OF_DICTIONARIES_WITH_SERIES_TO_ANALYZE, obtain_boxplot_with_filled_boxes = OBTAIN_BOXPLOT_WITH_FILLED_BOXES, obtain_boxplot_with_notched_boxes = OBTAIN_BOXPLOT_WITH_NOTCHED_BOXES, x_axis_rotation = X_AXIS_ROTATION, y_axis_rotation = Y_AXIS_ROTATION, grid = GRID, horizontal_axis_title = HORIZONTAL_AXIS_TITLE, vertical_axis_title = VERTICAL_AXIS_TITLE, plot_title = PLOT_TITLE, export_png = EXPORT_PNG, directory_to_save = DIRECTORY_TO_SAVE, file_name = FILE_NAME, png_resolution_dpi = PNG_RESOLUTION_DPI)

### **Performing AB-Tests**
- Compare if different groups present significant difference of means or proportions using t-test.
- For comparison of proportions of success on binary (categorical) results, the compared column must have been labelled with 0 for failure and 1 for success.
- Only compare 2 series.

In [ ]:
WHAT_TO_COMPARE = 'mean'
# WHAT_TO_COMPARE = 'mean' for comparing differences between mean values.
# WHAT_TO_COMPARE = 'proportion' for comparing differences between proportions.

CONFIDENCE_LEVEL_PERCENT = 95
# CONFIDENCE_LEVEL_PERCENT = 95 = 95% confidence
# It is the percent of confidence for the analysis.
# Set CONFIDENCE_LEVEL_PERCENT = 90 to get 0.90 = 90% confidence in the analysis.
# Notice that, when less trust is needed, we can reduce confidence_level_pct
# to get less restrictive results.


DATA_IN_SAME_COLUMN = False

# Parameters to input when DATA_IN_SAME_COLUMN = True:
DATASET = None #Alternatively: object containing the dataset to be analyzed (e.g. DATASET = dataset)
COLUMN_WITH_LABELS_OR_GROUPS = None # Alternatively: correct name for X-column
VARIABLE_TO_ANALYZE = None # Alternatively: correct name for Y-column

# DATA_IN_SAME_COLUMN = False: set as True if all the values to plot are in a same column.
# If DATA_IN_SAME_COLUMN = True, you must specify the dataframe containing the data as DATASET;
# the column containing the label or group indication as COLUMN_WITH_LABELS_OR_GROUPS; and the column 
# containing the variable to analyze as VARIABLE_TO_ANALYZE.

# If COLUMN_WITH_LABELS_OR_GROUPS is None, the analysis will not be performed.

# DATASET is an object, so do not declare it in quotes. The other three arguments (columns' names) 
# are strings, so declare in quotes.     
# Example: suppose you have a dataframe saved as dataset, and two groups A and B to compare. 
# All the results for both groups are in a column named 'results'. If the result is for
# an entry from group A, then a column named 'group' has the value 'A'. If it is for group B,
# column 'group' shows the value 'B'. In this example:
# DATA_IN_SAME_COLUMN = True,
# DATASET = dataset,
# COLUMN_WITH_LABELS_OR_GROUPS = 'group',
# VARIABLE_TO_ANALYZE = 'results'.


# If you want to declare a list of dictionaries, keep DATA_IN_SAME_COLUMN = False and keep
# DATASET = None (the other arguments may be set as None, but it is not mandatory: 
# COLUMN_WITH_LABELS_OR_GROUPS = None, VARIABLE_TO_ANALYZE = None).
    
# Parameter to input when DATA_IN_SAME_COLUMN = False:
LIST_OF_DICTIONARIES_WITH_SERIES_TO_ANALYZE = [
    
    {'values_to_analyze': None, 'label': None}, 
    {'values_to_analyze': None, 'label': None}
    
]
# LIST_OF_DICTIONARIES_WITH_SERIES_TO_ANALYZE: {'values_to_analyze': None, 'label': None}
# if data is already converted to series, lists or arrays, provide them as a list of dictionaries. 
# It must be declared as a list, in brackets, even if there is a single dictionary.
# Use always the same keys: 'values_to_analyze' for values that will be analyzed, and 'label' for
# the label or group correspondent to the series (may be a number or a string). 
# If you do not want to declare a series, simply keep as None, but do not remove or rename a 
# key (ALWAYS USE THE KEYS SHOWN AS MODEL).
# If you want, you can remove elements (dictionaries) from the list to declare fewer elements;
# and you can also add more elements (dictionaries) to the lists, if you need to plot more series.
# Simply put a comma after the last element from the list and declare a new dictionary, keeping the
# same keys: {'values_to_analyze': y, 'label': 'series_y'}, where y represents the values
# to analyze, and 'series_y' is the label 
# (you can pass 'label': None, but if values_to_analyze' is None, the new 
# dictionary will be ignored).
    
# Example:
# list_of_dictionaries_with_series_to_analyze = 
# [{'values_to_analyze': DATASET['Y1'], 'label': 'label1'}, 
# {'values_to_analyze': DATASET['Y2'], 'label': 'label2'}]
# will test two series, Y1 and Y2.
# Always use only two dictionaries

    
# Dictionary storing full analysis returned as summary_dict
# Simply modify this object on the left of equality:
summary_dict = AB_testing (what_to_compare = WHAT_TO_COMPARE, confidence_level_pct = CONFIDENCE_LEVEL_PERCENT, data_in_same_column = DATA_IN_SAME_COLUMN, df = DATASET, column_with_labels_or_groups = COLUMN_WITH_LABELS_OR_GROUPS, variable_to_analyze = VARIABLE_TO_ANALYZE, list_of_dictionaries_with_series_to_analyze = LIST_OF_DICTIONARIES_WITH_SERIES_TO_ANALYZE)

### **Importing or exporting models and dictionaries (or lists)**

#### Case 1: import only a model

In [ ]:
ACTION = 'import'
# ACTION = 'import' for importing a model and/or a dictionary;
# ACTION = 'export' for exporting a model and/or a dictionary.

OBJECTS_MANIPULATED = 'model_only'
# OBJECTS_MANIPULATED = 'model_only' if only a model will be manipulated.
# OBJECTS_MANIPULATED = 'dict_or_list_only' if only a dictionary will be manipulated.
# OBJECTS_MANIPULATED = 'model_and_dict' if both a model and a dictionary will 
#  be manipulated.

MODEL_FILE_NAME = None
# MODEL_FILE_NAME: string with the name of the file containing the model (for 'import');
# or of the name that the exported file will have (for 'export')
# e.g. MODEL_FILE_NAME = 'model'
# WARNING: Do not add the file extension.
# Keep it in quotes. Keep MODEL_FILE_NAME = None if no model will be manipulated.

DICTIONARY_OR_LIST_FILE_NAME = None
# DICTIONARY_OR_LIST_FILE_NAME: string with the name of the file containing the dictionary 
# (for 'import');
# or of the name that the exported file will have (for 'export')
# e.g. DICTIONARY_OR_LIST_FILE_NAME = 'history_dict'
# WARNING: Do not add the file extension.
# Keep it in quotes. Keep DICTIONARY_OR_LIST_FILE_NAME = None if no dictionary 
# or list will be manipulated.

DIRECTORY_PATH = ''
# DIRECTORY_PATH: path of the directory where the model will be saved,
# or from which the model will be retrieved. If no value is provided,
# the DIRECTORY_PATH will be the root: ""
# Notice that the model and the dictionary must be stored in the same path.
# If a model and a dictionary will be exported, they will be stored in the same
# DIRECTORY_PATH.
    
MODEL_TYPE = 'arima'
# This parameter has effect only when a model will be manipulated.
# MODEL_TYPE: 'keras' for deep learning Keras/ TensorFlow models with extension .keras
# MODEL_TYPE: 'tensorflow_general' for generic deep learning tensorflow models containing 
# custom layers, losses and architectures. Such models are compressed as tar.gz, tar, or zip.
# MODEL_TYPE = 'sklearn' for models from Scikit-learn (non-deep learning)
# MODEL_TYPE = 'xgb_regressor' for XGBoost regression models (non-deep learning)
# MODEL_TYPE = 'xgb_classifier' for XGBoost classification models (non-deep learning)
# MODEL_TYPE = 'arima' for ARIMA model (Statsmodels)
# MODEL_TYPE = 'prophet' for Facebook Prophet model
# MODEL_TYPE = 'anomaly_detector' for the Anomaly Detection model

DICT_OR_LIST_TO_EXPORT = None
MODEL_TO_EXPORT = None 
# These two parameters have effect only when ACTION == 'export'. In this case, they
# must be declared. If ACTION == 'export', keep:
# DICT_OR_LIST_TO_EXPORT = None, 
# MODEL_TO_EXPORT = None
# If one of these objects will be exported, substitute None by the name of the object
# e.g. if your model is stored in the global memory as 'keras_model' declare:
# MODEL_TO_EXPORT = keras_model. Notice that it must be declared without quotes, since
# it is not a string, but an object.
# For exporting a dictionary named as 'dict':
# DICT_OR_LIST_TO_EXPORT = dict

USE_COLAB_MEMORY = False
# USE_COLAB_MEMORY: this parameter has only effect when using Google Colab (or it will
# raise an error). Set as USE_COLAB_MEMORY = True if you want to use the instant memory
# from Google Colaboratory: you will update or download the file and it will be available
# only during the time when the kernel is running. It will be excluded when the kernel
# dies, for instance, when you close the notebook.
    
# If ACTION == 'export' and USE_COLAB_MEMORY == True, then the file will be downloaded
# to your computer (running the cell will start the download).

# Model object saved as model.
# Simply modify this object on the left of equality:
model = import_export_model_list_dict (action = ACTION, objects_manipulated = OBJECTS_MANIPULATED, model_file_name = MODEL_FILE_NAME, dictionary_or_list_file_name = DICTIONARY_OR_LIST_FILE_NAME, directory_path = DIRECTORY_PATH, model_type = MODEL_TYPE, dict_or_list_to_export = DICT_OR_LIST_TO_EXPORT, model_to_export = MODEL_TO_EXPORT, use_colab_memory = USE_COLAB_MEMORY) 

#### Case 2: import only a dictionary or a list

In [ ]:
ACTION = 'import'
# ACTION = 'import' for importing a model and/or a dictionary;
# ACTION = 'export' for exporting a model and/or a dictionary.

OBJECTS_MANIPULATED = 'dict_or_list_only'
# OBJECTS_MANIPULATED = 'model_only' if only a model will be manipulated.
# OBJECTS_MANIPULATED = 'dict_or_list_only' if only a dictionary will be manipulated.
# OBJECTS_MANIPULATED = 'model_and_dict' if both a model and a dictionary will 
#  be manipulated.

MODEL_FILE_NAME = None
# MODEL_FILE_NAME: string with the name of the file containing the model (for 'import');
# or of the name that the exported file will have (for 'export')
# e.g. MODEL_FILE_NAME = 'model'
# WARNING: Do not add the file extension.
# Keep it in quotes. Keep MODEL_FILE_NAME = None if no model will be manipulated.

DICTIONARY_OR_LIST_FILE_NAME = None
# DICTIONARY_OR_LIST_FILE_NAME: string with the name of the file containing the dictionary 
# (for 'import');
# or of the name that the exported file will have (for 'export')
# e.g. DICTIONARY_OR_LIST_FILE_NAME = 'history_dict'
# WARNING: Do not add the file extension.
# Keep it in quotes. Keep DICTIONARY_OR_LIST_FILE_NAME = None if no dictionary 
# or list will be manipulated.

DIRECTORY_PATH = ''
# DIRECTORY_PATH: path of the directory where the model will be saved,
# or from which the model will be retrieved. If no value is provided,
# the DIRECTORY_PATH will be the root: ""
# Notice that the model and the dictionary must be stored in the same path.
# If a model and a dictionary will be exported, they will be stored in the same
# DIRECTORY_PATH.
    
MODEL_TYPE = 'arima'
# This parameter has effect only when a model will be manipulated.
# MODEL_TYPE: 'keras' for deep learning Keras/ TensorFlow models with extension .keras
# MODEL_TYPE: 'tensorflow_general' for generic deep learning tensorflow models containing 
# custom layers, losses and architectures. Such models are compressed as tar.gz, tar, or zip.
# MODEL_TYPE = 'sklearn' for models from Scikit-learn (non-deep learning)
# MODEL_TYPE = 'xgb_regressor' for XGBoost regression models (non-deep learning)
# MODEL_TYPE = 'xgb_classifier' for XGBoost classification models (non-deep learning)
# MODEL_TYPE = 'arima' for ARIMA model (Statsmodels)
# MODEL_TYPE = 'prophet' for Facebook Prophet model
# MODEL_TYPE = 'anomaly_detector' for the Anomaly Detection model

DICT_OR_LIST_TO_EXPORT = None
MODEL_TO_EXPORT = None 
# These two parameters have effect only when ACTION == 'export'. In this case, they
# must be declared. If ACTION == 'export', keep:
# DICT_OR_LIST_TO_EXPORT = None, 
# MODEL_TO_EXPORT = None
# If one of these objects will be exported, substitute None by the name of the object
# e.g. if your model is stored in the global memory as 'keras_model' declare:
# MODEL_TO_EXPORT = keras_model. Notice that it must be declared without quotes, since
# it is not a string, but an object.
# For exporting a dictionary named as 'dict':
# DICT_OR_LIST_TO_EXPORT = dict

USE_COLAB_MEMORY = False
# USE_COLAB_MEMORY: this parameter has only effect when using Google Colab (or it will
# raise an error). Set as USE_COLAB_MEMORY = True if you want to use the instant memory
# from Google Colaboratory: you will update or download the file and it will be available
# only during the time when the kernel is running. It will be excluded when the kernel
# dies, for instance, when you close the notebook.
    
# If ACTION == 'export' and USE_COLAB_MEMORY == True, then the file will be downloaded
# to your computer (running the cell will start the download).

# Dictionary or list saved as imported_dict_or_list.
# Simply modify this object on the left of equality:
imported_dict_or_list = import_export_model_list_dict (action = ACTION, objects_manipulated = OBJECTS_MANIPULATED, model_file_name = MODEL_FILE_NAME, dictionary_or_list_file_name = DICTIONARY_OR_LIST_FILE_NAME, directory_path = DIRECTORY_PATH, model_type = MODEL_TYPE, dict_or_list_to_export = DICT_OR_LIST_TO_EXPORT, model_to_export = MODEL_TO_EXPORT, use_colab_memory = USE_COLAB_MEMORY) 

#### Case 3: import a model and a dictionary (or a list)

In [ ]:
ACTION = 'import'
# ACTION = 'import' for importing a model and/or a dictionary;
# ACTION = 'export' for exporting a model and/or a dictionary.

OBJECTS_MANIPULATED = 'model_and_dict'
# OBJECTS_MANIPULATED = 'model_only' if only a model will be manipulated.
# OBJECTS_MANIPULATED = 'dict_or_list_only' if only a dictionary will be manipulated.
# OBJECTS_MANIPULATED = 'model_and_dict' if both a model and a dictionary will 
#  be manipulated.

MODEL_FILE_NAME = None
# MODEL_FILE_NAME: string with the name of the file containing the model (for 'import');
# or of the name that the exported file will have (for 'export')
# e.g. MODEL_FILE_NAME = 'model'
# WARNING: Do not add the file extension.
# Keep it in quotes. Keep MODEL_FILE_NAME = None if no model will be manipulated.

DICTIONARY_OR_LIST_FILE_NAME = None
# DICTIONARY_OR_LIST_FILE_NAME: string with the name of the file containing the dictionary 
# (for 'import');
# or of the name that the exported file will have (for 'export')
# e.g. DICTIONARY_OR_LIST_FILE_NAME = 'history_dict'
# WARNING: Do not add the file extension.
# Keep it in quotes. Keep DICTIONARY_OR_LIST_FILE_NAME = None if no dictionary 
# or list will be manipulated.

DIRECTORY_PATH = ''
# DIRECTORY_PATH: path of the directory where the model will be saved,
# or from which the model will be retrieved. If no value is provided,
# the DIRECTORY_PATH will be the root: ""
# Notice that the model and the dictionary must be stored in the same path.
# If a model and a dictionary will be exported, they will be stored in the same
# DIRECTORY_PATH.
    
MODEL_TYPE = 'arima'
# This parameter has effect only when a model will be manipulated.
# MODEL_TYPE: 'keras' for deep learning Keras/ TensorFlow models with extension .keras
# MODEL_TYPE: 'tensorflow_general' for generic deep learning tensorflow models containing 
# custom layers, losses and architectures. Such models are compressed as tar.gz, tar, or zip.
# MODEL_TYPE = 'sklearn' for models from Scikit-learn (non-deep learning)
# MODEL_TYPE = 'xgb_regressor' for XGBoost regression models (non-deep learning)
# MODEL_TYPE = 'xgb_classifier' for XGBoost classification models (non-deep learning)
# MODEL_TYPE = 'arima' for ARIMA model (Statsmodels)
# MODEL_TYPE = 'prophet' for Facebook Prophet model
# MODEL_TYPE = 'anomaly_detector' for the Anomaly Detection model

DICT_OR_LIST_TO_EXPORT = None
MODEL_TO_EXPORT = None 
# These two parameters have effect only when ACTION == 'export'. In this case, they
# must be declared. If ACTION == 'export', keep:
# DICT_OR_LIST_TO_EXPORT = None, 
# MODEL_TO_EXPORT = None
# If one of these objects will be exported, substitute None by the name of the object
# e.g. if your model is stored in the global memory as 'keras_model' declare:
# MODEL_TO_EXPORT = keras_model. Notice that it must be declared without quotes, since
# it is not a string, but an object.
# For exporting a dictionary named as 'dict':
# DICT_OR_LIST_TO_EXPORT = dict

USE_COLAB_MEMORY = False
# USE_COLAB_MEMORY: this parameter has only effect when using Google Colab (or it will
# raise an error). Set as USE_COLAB_MEMORY = True if you want to use the instant memory
# from Google Colaboratory: you will update or download the file and it will be available
# only during the time when the kernel is running. It will be excluded when the kernel
# dies, for instance, when you close the notebook.
    
# If ACTION == 'export' and USE_COLAB_MEMORY == True, then the file will be downloaded
# to your computer (running the cell will start the download).

# Model object saved as model. Dictionary or list saved as imported_dict_or_list.
# Simply modify these objects on the left of equality:
model, imported_dict_or_list = import_export_model_list_dict (action = ACTION, objects_manipulated = OBJECTS_MANIPULATED, model_file_name = MODEL_FILE_NAME, dictionary_or_list_file_name = DICTIONARY_OR_LIST_FILE_NAME, directory_path = DIRECTORY_PATH, model_type = MODEL_TYPE, dict_or_list_to_export = DICT_OR_LIST_TO_EXPORT, model_to_export = MODEL_TO_EXPORT, use_colab_memory = USE_COLAB_MEMORY) 

#### Case 4: export a model and/or a dictionary (or a list)

In [ ]:
ACTION = 'export'
# ACTION = 'import' for importing a model and/or a dictionary;
# ACTION = 'export' for exporting a model and/or a dictionary.

OBJECTS_MANIPULATED = 'model_only'
# OBJECTS_MANIPULATED = 'model_only' if only a model will be manipulated.
# OBJECTS_MANIPULATED = 'dict_or_list_only' if only a dictionary will be manipulated.
# OBJECTS_MANIPULATED = 'model_and_dict' if both a model and a dictionary will 
#  be manipulated.

MODEL_FILE_NAME = None
# MODEL_FILE_NAME: string with the name of the file containing the model (for 'import');
# or of the name that the exported file will have (for 'export')
# e.g. MODEL_FILE_NAME = 'model'
# WARNING: Do not add the file extension.
# Keep it in quotes. Keep MODEL_FILE_NAME = None if no model will be manipulated.

DICTIONARY_OR_LIST_FILE_NAME = None
# DICTIONARY_OR_LIST_FILE_NAME: string with the name of the file containing the dictionary 
# (for 'import');
# or of the name that the exported file will have (for 'export')
# e.g. DICTIONARY_OR_LIST_FILE_NAME = 'history_dict'
# WARNING: Do not add the file extension.
# Keep it in quotes. Keep DICTIONARY_OR_LIST_FILE_NAME = None if no dictionary 
# or list will be manipulated.

DIRECTORY_PATH = ''
# DIRECTORY_PATH: path of the directory where the model will be saved,
# or from which the model will be retrieved. If no value is provided,
# the DIRECTORY_PATH will be the root: ""
# Notice that the model and the dictionary must be stored in the same path.
# If a model and a dictionary will be exported, they will be stored in the same
# DIRECTORY_PATH.
    
MODEL_TYPE = 'arima'
# This parameter has effect only when a model will be manipulated.
# MODEL_TYPE: 'keras' for deep learning Keras/ TensorFlow models with extension .keras
# MODEL_TYPE: 'tensorflow_general' for generic deep learning tensorflow models containing 
# custom layers, losses and architectures. Such models are compressed as tar.gz, tar, or zip.
# MODEL_TYPE = 'sklearn' for models from Scikit-learn (non-deep learning)
# MODEL_TYPE = 'xgb_regressor' for XGBoost regression models (non-deep learning)
# MODEL_TYPE = 'xgb_classifier' for XGBoost classification models (non-deep learning)
# MODEL_TYPE = 'arima' for ARIMA model (Statsmodels)
# MODEL_TYPE = 'prophet' for Facebook Prophet model
# MODEL_TYPE = 'anomaly_detector' for the Anomaly Detection model

DICT_OR_LIST_TO_EXPORT = None
MODEL_TO_EXPORT = None 
# These two parameters have effect only when ACTION == 'export'. In this case, they
# must be declared. If ACTION == 'export', keep:
# DICT_OR_LIST_TO_EXPORT = None, 
# MODEL_TO_EXPORT = None
# If one of these objects will be exported, substitute None by the name of the object
# e.g. if your model is stored in the global memory as 'keras_model' declare:
# MODEL_TO_EXPORT = keras_model. Notice that it must be declared without quotes, since
# it is not a string, but an object.
# For exporting a dictionary named as 'dict':
# DICT_OR_LIST_TO_EXPORT = dict

USE_COLAB_MEMORY = False
# USE_COLAB_MEMORY: this parameter has only effect when using Google Colab (or it will
# raise an error). Set as USE_COLAB_MEMORY = True if you want to use the instant memory
# from Google Colaboratory: you will update or download the file and it will be available
# only during the time when the kernel is running. It will be excluded when the kernel
# dies, for instance, when you close the notebook.
    
# If ACTION == 'export' and USE_COLAB_MEMORY == True, then the file will be downloaded
# to your computer (running the cell will start the download).

import_export_model_list_dict (action = ACTION, objects_manipulated = OBJECTS_MANIPULATED, model_file_name = MODEL_FILE_NAME, dictionary_or_list_file_name = DICTIONARY_OR_LIST_FILE_NAME, directory_path = DIRECTORY_PATH, model_type = MODEL_TYPE, dict_or_list_to_export = DICT_OR_LIST_TO_EXPORT, model_to_export = MODEL_TO_EXPORT, use_colab_memory = USE_COLAB_MEMORY) 

## **Exporting the dataframe as CSV file (to notebook's workspace)**

In [ ]:
## WARNING: all files exported from this function are .csv (comma separated values)

DATAFRAME_OBJ_TO_BE_EXPORTED = dataset
# Alternatively: object containing the dataset to be exported.
# DATAFRAME_OBJ_TO_BE_EXPORTED: dataframe object that is going to be exported from the
# function. Since it is an object (not a string), it should not be declared in quotes.
# example: DATAFRAME_OBJ_TO_BE_EXPORTED = dataset will export the dataset object.
# ATTENTION: The dataframe object must be a Pandas dataframe.

FILE_DIRECTORY_PATH = ""
# FILE_DIRECTORY_PATH - (string, in quotes): input the path of the directory 
# (e.g. folder path) where the file is stored. e.g. FILE_DIRECTORY_PATH = "" 
# or FILE_DIRECTORY_PATH = "folder"
# If you want to export the file to AWS S3, this parameter will have no effect.
# In this case, you can set FILE_DIRECTORY_PATH = None

NEW_FILE_NAME_WITHOUT_EXTENSION = "dataset"
# NEW_FILE_NAME_WITHOUT_EXTENSION - (string, in quotes): input the name of the 
# file without the extension. e.g. set NEW_FILE_NAME_WITHOUT_EXTENSION = "my_file" 
# to export the CSV file 'my_file.csv' to notebook's workspace.

export_pd_dataframe_as_csv (dataframe_obj_to_be_exported = DATAFRAME_OBJ_TO_BE_EXPORTED, new_file_name_without_extension = NEW_FILE_NAME_WITHOUT_EXTENSION, file_directory_path = FILE_DIRECTORY_PATH)

## **Exporting dataframes as Excel file tables**

In [ ]:
## WARNING: all files exported from this function are .xlsx

FILE_NAME_WITHOUT_EXTENSION = "datasets"
# (string, in quotes): input the name of the 
# file without the extension. e.g. new_file_name_without_extension = "my_file" 
# will export a file 'my_file.xlsx' to notebook's workspace.

EXPORTED_TABLES = [{'dataframe_obj_to_be_exported': None, 
                    'excel_sheet_name': None},]

# exported_tables is a list of dictionaries. User may declare several dictionaries, 
# as long as the keys are always the same, and if the values stored in keys are not None.
      
# key 'dataframe_obj_to_be_exported': dataframe object that is going to be exported from the
# function. Since it is an object (not a string), it should not be declared in quotes.
# example: dataframe_obj_to_be_exported = dataset will export the dataset object.
# ATTENTION: The dataframe object must be a Pandas dataframe.

# key 'excel_sheet_name': string containing the name of the sheet to be written on the
# exported Excel file. Example: excel_sheet_name = 'tab_1' will save the dataframe in the
# sheet 'tab_1' from the file named as file_name_without_extension.

# examples: exported_tables = [{'dataframe_obj_to_be_exported': dataset1, 
# 'excel_sheet_name': 'sheet1'},]
# will export only dataset1 as 'sheet1';
# exported_tables = [{'dataframe_obj_to_be_exported': dataset1, 'excel_sheet_name': 'sheet1'},
# {'dataframe_obj_to_be_exported': dataset2, 'excel_sheet_name': 'sheet2']
# will export dataset1 as 'sheet1' and dataset2 as 'sheet2'.

# Notice that if the file does not contain the exported sheets, they will be created. If it has,
# the sheets will be replaced.

FILE_DIRECTORY_PATH = ""
# FILE_DIRECTORY_PATH - (string, in quotes): input the path of the directory 
# (e.g. folder path) where the file is stored. e.g. FILE_DIRECTORY_PATH = "" 
# or FILE_DIRECTORY_PATH = "folder"
# If you want to export the file to AWS S3, this parameter will have no effect.
# In this case, you can set FILE_DIRECTORY_PATH = None


export_pd_dataframe_as_excel (file_name_without_extension = FILE_NAME_WITHOUT_EXTENSION, exported_tables = EXPORTED_TABLES, file_directory_path = FILE_DIRECTORY_PATH)

****

## IDSW v2.0.0 Additions: Non-Parametric Tests, Proportions, Monte Carlo, Sampling, Bayesian Inference & Plotly (plotly=True)

In [ ]:
# Non-parametric ANOVA (Kruskal-Wallis) with Plotly
anova_df = anova_box_violin_plot(plot_type='violin', data_in_same_column=True, df=DATASET, column_with_labels_or_groups='GROUP_COL', variable_to_analyze='VALUE_COL', non_parametric=True, plotly=True)

# Hypothesis Test for Proportions
prop_summary, prop_df = test_proportion_diffs(df=DATASET, group_col='GROUP_COL', target_col='BINARY_COL', non_parametric=False)

# Monte Carlo Simulation with equation string & Plotly
mc_df, mc_summary = monte_carlo_simulation(simulation_function_or_equation='x * y - z', input_distributions={'x': ('normal', 100, 10), 'y': ('uniform', 0.8, 1.2), 'z': ('normal', 20, 3)}, n_iterations=5000, plotly=True)

# Bootstrap Sampling Distribution & Distribution Fitting with Plotly
samp_stats, samp_summary = generate_sampling_distribution(df=DATASET, column_to_analyze='VALUE_COL', statistic='mean', n_replications=1000, plotly=True)
gen_df, fit_summary = generate_and_fit_distribution(distribution_name='lognormal', sample_size=1000, plotly=True)